# Traffic Congestion Prediction: MobileNet + Echo State Network (ESN)

This notebook implements a **training-free** pipeline for video classification, suitable for cloud environments (Kaggle/Colab).

### Approach (Option 4):
1.  **Spatial Features**: Use a **Frozen Pre-trained MobileNetV2** to extract high-level visual features (1280-dim) from each frame. No backpropagation is performed on the CNN.
2.  **Temporal Features**: Use an **Echo State Network (ESN)** (Reservoir Computing) to model the temporal evolution of these features over time.
3.  **Classification**: Train a linear readout (Ridge Regression) to predict traffic congestion levels.

**Advantages**: 
- **Extremely Fast Training**: No gradient descent, just one-shot matrix solution.
- **Low Compute**: Can perform reasonably well without high-end GPUs for training.
- **Video-Native**: Processes raw video frames.

In [1]:
# Install dependencies
!pip install opencv-python-headless scikit-learn polars

## 1. Configuration & Paths
Please set the following paths to your dataset locations.

In [19]:
# --- USER CONFIGURATION ---
BASE_DIR = '/teamspace/studios/this_studio/Barbados_Traffic_Analysis_Challenge_dev' # Example
VIDEO_DIR = '/teamspace/studios/this_studio/videos' # Example
TRAIN_CSV = os.path.join(BASE_DIR, 'demos/Train.csv')
TEST_CSV = os.path.join(BASE_DIR, 'demos/TestInputSegments.csv')
SAMPLE_SUB = os.path.join(BASE_DIR, 'demos/SampleSubmission.csv')

# Model Config
IMG_SIZE = (224, 224)
SEQ_LENGTH = 30         # Number of frames to extract per video (downsampled)
RESERVOIR_DIM = 1000    # ESN Reservoir Size
SPECTRAL_RADIUS = 0.9
LEAK_RATE = 0.2
RIDGE_ALPHA = 1.0
# --------------------------

In [10]:
import os
from google.oauth2 import service_account

# If you uploaded the file to a dataset:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/teamspace/studios/this_studio/tokens.json"


In [28]:
client = storage.Client(project="brb-traffic")

# Base directories and bucket name
bucket_name = 'brb-traffic'

# Video paths
video_dir = VIDEO_DIR
video_path = '/teamspace/studios/this_studio/videos'
os.makedirs(video_dir, exist_ok=True)

# Datasheet paths
train_csv_path = TRAIN_CSV
sample_submission_csv_path = SAMPLE_SUB

In [20]:
# Load the dataset
train = pd.read_csv(TRAIN_CSV)

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

(16076, 14)

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [23]:
ss = pd.read_csv(SAMPLE_SUB)
display(ss.shape,ss.head())

(880, 3)

,ID,Target,Target_Accuracy
0,time_segment_129_Norman Niles #1_congestion_en...,free flowing,free flowing
1,time_segment_130_Norman Niles #1_congestion_en...,heavy delay,heavy delay
2,time_segment_131_Norman Niles #1_congestion_en...,free flowing,free flowing
3,time_segment_132_Norman Niles #1_congestion_en...,heavy delay,heavy delay
4,time_segment_133_Norman Niles #1_congestion_en...,free flowing,free flowing


In [26]:
# %%capture
blobs=train.videos.tolist()[0:2000]
print(f"Number of blobs selected: {len(blobs)}")
display(blobs)

Number of blobs selected: 2000


['normanniles1/normanniles1_2025-10-20-06-00-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-01-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-02-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-03-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-04-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-05-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-06-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-07-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-08-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-09-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-10-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-11-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-12-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-13-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-14-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-15-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-16-45.mp4',
 'normanniles1/normanniles1_2025-10-20-06-17-45.mp4',
 'normanniles1/normanniles1_

In [29]:
# %%capture
from google.api_core.exceptions import NotFound

print(f"--- Debugging Blobs Variable ---")
print(f"Type of 'blobs' before loop: {type(blobs)}")
print(f"Content of 'blobs' (first 5): {blobs[:5]}")
print(f"----------------------------------")

# --- Existing loop for downloading files from the 'blobs' list ---
for blob_name in blobs:
    # Ensure blob_name is a string before proceeding
    if not isinstance(blob_name, str):
        print(f"❌ Error: Expected string for blob_name, but got {type(blob_name)}. Skipping.")
        continue

    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print(f"Attempting to download blob: '{blob_name}' to '{local_path}'") # Added print for clarity
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading '{blob_name}': {e}")

--- Debugging Blobs Variable ---
Type of 'blobs' before loop: <class 'list'>
Content of 'blobs' (first 5): ['normanniles1/normanniles1_2025-10-20-06-00-45.mp4', 'normanniles1/normanniles1_2025-10-20-06-01-45.mp4', 'normanniles1/normanniles1_2025-10-20-06-02-45.mp4', 'normanniles1/normanniles1_2025-10-20-06-03-45.mp4', 'normanniles1/normanniles1_2025-10-20-06-04-45.mp4']
----------------------------------
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-00-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-00-45.mp4'
✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-00-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-01-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-01-45.mp4'
✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-01-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-02-45.mp4' to '/

✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-10-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-11-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-11-45.mp4'
✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-11-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-12-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-12-45.mp4'
✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-12-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-13-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-13-45.mp4'
✅ Downloaded to: /teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06-13-45.mp4
Attempting to download blob: 'normanniles1/normanniles1_2025-10-20-06-14-45.mp4' to '/teamspace/studios/this_studio/videos/normanniles1_2025-10-20-06

## 2. Feature Extraction (MobileNetV2)
We load a MobileNetV2 pre-trained on ImageNet, remove the top classification layer, and use global average pooling to get a 1280-dimensional vector for each frame.

In [30]:
def build_feature_extractor():
    base_model = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        pooling='avg',
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    base_model.trainable = False  # Freeze weights
    return base_model

feat_extractor = build_feature_extractor()
print("Feature Extractor Loaded: MobileNetV2 (Frozen)")

I0000 00:00:1768921408.049667    8210 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:1e.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Feature Extractor Loaded: MobileNetV2 (Frozen)


## 3. Video Processing Utils
Functions to read video frames and extract features.

In [31]:
def get_video_features(video_path):
    frames = extract_frames(video_path)
    if len(frames) == 0:
        return np.zeros(1280) # Return zero vector if empty
    
    # Extract frame-level features: (T_frames, 1280)
    frame_features = feat_extractor.predict(frames, verbose=0)
    
    # Temporal Pooling: Average frame features to get one vector per video
    # Shape: (1280,)
    video_feature = np.mean(frame_features, axis=0)
    return video_feature

## 4. Echo State Network Class
A pure NumPy implementation of ESN for sequence classification.

In [ ]:
def create_sequential_blocks(df, is_test=False):
    """
    Groups a dataframe into sequential blocks of videos based on time_segment_id.
    Returns: List of (X_block, y_block) tuples.
    X_block: (T_videos, 1280)
    y_block: (T_videos, )
    """
    blocks = []
    
    # Ensure sorted by view/camera and time
    # Assuming 'time_segment_id' implies order. If datetime is available, sort by it.
    if 'datetimestamp_start' in df.columns:
        df['dt'] = pd.to_datetime(df['datetimestamp_start'])
        df = df.sort_values(by=['view_label', 'dt'])
    else:
        df = df.sort_values(by=['view_label', 'time_segment_id'])
    
    print(f"Grouping {len(df)} samples into sequential blocks...")
    
    grouped = df.groupby('view_label')
    
    for view_id, group in grouped:
        # Identify continuity breaks
        # We assume time_segment_id is continuous integers (0, 1, 2...) for continuous time
        # diff != 1 implies a break in the sequence
        group = group.copy()
        group['block_id'] = (group['time_segment_id'].diff() != 1).cumsum()
        
        for _, block in group.groupby('block_id'):
            # Stack features
            if 'features' not in block.columns:
                continue 
                
            X_seq = np.stack(block['features'].values)
            
            if not is_test:
                y_seq = block['label_code'].values
            else:
                y_seq = np.zeros(len(block)) # Dummy labels for test
                
            blocks.append((X_seq, y_seq, block['video_full_path'].values if is_test else None))
            
    print(f"Refined into {len(blocks)} blocks.")
    return blocks

In [32]:
class SequentialESN:
    """ESN that processes sequences of videos (inter-video modeling)."""
    def __init__(self, input_dim=1280, res_dim=2000, rho=0.9, leak=0.2, alpha=1.0):
        self.res_dim = res_dim
        self.rho = rho
        self.leak = leak
        self.alpha = alpha
        
        rng = np.random.RandomState(42)
        self.W_in = rng.uniform(-1, 1, (res_dim, input_dim))
        
        # Sparse recurrent weights
        self.W_res = rng.uniform(-1, 1, (res_dim, res_dim))
        mask = rng.rand(res_dim, res_dim) > 0.95
        self.W_res[mask] = 0
        
        # Spectral Norm
        try:
            eigenvalues = np.linalg.eigvals(self.W_res)
            max_eig = np.max(np.abs(eigenvalues))
            self.W_res *= (self.rho / max_eig)
        except:
            self.W_res *= 0.9 # Fallback
        
        self.readout = Ridge(alpha=self.alpha)
        
    def get_states_sequence(self, input_seq):
        """
        Processes a sequence of inputs (T, Input) and returns sequence of states (T, Res).
        State is carried over from t to t+1.
        """
        T = input_seq.shape[0]
        states = np.zeros((T, self.res_dim))
        x = np.zeros(self.res_dim)
        
        for t in range(T):
            u = input_seq[t]
            # x(t) = (1-a)x(t-1) + a*tanh(Win*u + Wres*x(t-1))
            pre = np.dot(self.W_in, u) + np.dot(self.W_res, x)
            update = np.tanh(pre)
            x = (1 - self.leak) * x + self.leak * update
            states[t] = x
            
        return states
    
    def fit(self, blocks):
        """
        blocks: List of (X_seq, y_seq, ...)
        Concatenates all timesteps from all blocks to train the linear readout.
        """
        all_states = []
        all_targets = []
        
        print(f"Training ESN on {len(blocks)} blocks...")
        for X_seq, y_seq, _ in blocks:
            # Get states for this block
            states_seq = self.get_states_sequence(X_seq)
            
            # Collect states and targets for every timestep
            all_states.append(states_seq)
            all_targets.append(y_seq)
            
        # Stack everything: (Total_Samples, Res_Dim)
        X_train_res = np.vstack(all_states)
        y_train_flat = np.concatenate(all_targets)
        
        print(f"Solving Ridge Regression on matrix {X_train_res.shape}...")
        self.readout.fit(X_train_res, y_train_flat)
        print("Done.")
        
    def predict(self, blocks):
        """
        Predicts for each timestep in each block.
        Returns list of prediction arrays.
        """
        all_preds = []
        
        for X_seq, _, _ in blocks:
            states_seq = self.get_states_sequence(X_seq)
            preds_seq = self.readout.predict(states_seq)
            all_preds.append(preds_seq)
            
        return all_preds

## 5. Load Data & Train
We will load video paths, labels, and run the pipeline.

In [ ]:
# --- 1. Load Data & Precompute Features ---

df_train = pd.read_csv(TRAIN_CSV)
df_train['video_full_path'] = df_train['videos'].apply(extract_camera_path)

congestion_map = {'free flowing': 0, 'light delay': 1, 'moderate delay': 2, 'heavy delay': 3}
df_train['label_code'] = df_train['congestion_enter_rating'].map(congestion_map).fillna(0).astype(int)

print("Extracting features for ALL Training videos...")
# Note: We simply call extracting function. We will store results back in DF.
# Optimizing: We can reuse the previous extract_dataset_features but simplified

feats_list = []
total = len(df_train)
for idx, row in df_train.iterrows():
    if idx % 100 == 0: print(f"{idx}/{total}")
    f = get_video_features(row['video_full_path'])
    feats_list.append(f)

df_train['features'] = feats_list

# --- 2. Sequential Grouping ---

# Create blocks
train_blocks = create_sequential_blocks(df_train)

# Split Blocks for Validation (Time-based split: Last 20% of BLOCKS)
n_val = int(len(train_blocks) * 0.2)
train_blocks_split = train_blocks[:-n_val]
val_blocks_split = train_blocks[-n_val:]
print(f"Train Blocks: {len(train_blocks_split)}, Val Blocks: {len(val_blocks_split)}")

# --- 3. Sequential ESN Training ---

esn = SequentialESN(res_dim=2000, leak=0.1) # Higher res_dim, lower leak for longer memory
esn.fit(train_blocks_split)

# --- 4. Validation ---
val_preds_list = esn.predict(val_blocks_split)

y_val_true = []
y_val_pred = []

for i, p_seq in enumerate(val_preds_list):
    _, t_seq, _ = val_blocks_split[i]
    y_val_true.extend(t_seq)
    y_val_pred.extend(p_seq)
    
y_val_pred_class = np.round(np.clip(y_val_pred, 0, 3)).astype(int)
print("Validation F1:", f1_score(y_val_true, y_val_pred_class, average='macro'))
print("Validation Acc:", accuracy_score(y_val_true, y_val_pred_class))

# --- 5. Inference (Test Blocks) ---

# Load Test - Assuming it maps to videos
df_test = pd.read_csv(TEST_CSV)
# NOTE: You must implement the mapping of test IDs to 'videos' path here if it's missing in TEST_CSV
# For now, assuming similar structure or user handles mapping
if 'videos' in df_test.columns:
    df_test['video_full_path'] = df_test['videos'].apply(extract_camera_path)
    print("Extracting Test Features...")
    test_feats = []
    for idx, row in df_test.iterrows():
        if idx % 100 == 0: print(f"{idx}/{len(df_test)}")
        test_feats.append(get_video_features(row['video_full_path']))
    df_test['features'] = test_feats
    
    test_blocks = create_sequential_blocks(df_test, is_test=True)
    print(f"Testing on {len(test_blocks)} blocks...")
    
    # Predict
    test_preds_list = esn.predict(test_blocks)
    
    # Flatten and Map to ID
    # We need to assume the blocks are returned in same order of creation, 
    # but we stored path/ID in the block tuple to be safe? 
    # The current create_sequential_blocks returns (X, y, paths) for test.
    # Ideally, we map back to the dataframe.
    
    print("Inference Done. (Implement mapping back to Submission CSV based on video/ID)")
else:
    print("Test CSV missing 'videos' column. Cannot run inference without video mapping.")

## 6. Inference on Test Set
This section generates the submission file.